# 🌿 AgriSmart AI — Day 3: Data Augmentation & Model Fine-Tuning

**Team:** Meet & Kruti (Core ML Track)

### 🎯 Goal Today:
1. **Heavy Data Augmentation** (rotation, zoom, brightness, flips) to handle real-field noisy photos.
2. **Fine-Tuning MobileNetV2** (unfreezing the top 20 layers with `Adam(learning_rate=1e-5)`).
3. **Compute Official Macro-F1 & Confusion Matrix** on the test set.
4. **Export final `agrismart_model.keras`**.

## ⚙️ Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn split-folders Pillow

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 GPU Detected: {gpus[0].name} (Training will take ~6-8 minutes!)")
else:
    print("⚠️ No GPU detected. Make sure to click: Runtime -> Change runtime type -> T4 GPU")

## 📁 Step 2: Mount Google Drive or Locate Dataset
If running in **Google Colab**, mount your Drive where the PlantVillage folder is stored.

In [ ]:
import os
import json
from pathlib import Path

# Mount Drive if running on Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    colab_env = True
except Exception:
    colab_env = False

# Check for pre-split data or raw dataset path
possible_split_dirs = [
    Path('data/split'),
    Path('/content/drive/MyDrive/AgriSmart-AI/data/split'),
    Path('/content/drive/MyDrive/data/split'),
]

DATA_SPLIT_DIR = next((p for p in possible_split_dirs if (p / 'train').exists()), None)

if DATA_SPLIT_DIR:
    print(f"✅ Found existing split dataset at: {DATA_SPLIT_DIR}")
else:
    print("⚠️ Split directory not found. Please provide path to raw 'color' dataset:")
    # Update this path if needed:
    RAW_DATASET_PATH = '/content/drive/MyDrive/plantvillage dataset/color'
    DATA_SPLIT_DIR = Path('data/split')
    import splitfolders
    print(f"Splitting raw dataset from {RAW_DATASET_PATH} into 70/15/15...")
    splitfolders.ratio(RAW_DATASET_PATH, output=str(DATA_SPLIT_DIR), seed=42, ratio=(0.70, 0.15, 0.15))

## 🔄 Step 3: Data Augmentation Pipeline
Apply random flips, rotations, zoom, and brightness variations so the model learns real-world leaf conditions.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=25,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    width_shift_range=0.15,
    height_shift_range=0.15,
    fill_mode='nearest'
)

eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    str(DATA_SPLIT_DIR / 'train'),
    target_size=(224, 224),
    batch_size=32,
    seed=42,
)

class_names = [name for name, index in sorted(train_data.class_indices.items(), key=lambda item: item[1])]

val_data = eval_datagen.flow_from_directory(
    str(DATA_SPLIT_DIR / 'val'),
    target_size=(224, 224),
    batch_size=32,
    classes=class_names,
    shuffle=False,
)

test_data = eval_datagen.flow_from_directory(
    str(DATA_SPLIT_DIR / 'test'),
    target_size=(224, 224),
    batch_size=32,
    classes=class_names,
    shuffle=False,
)

# Save class indices mapping
os.makedirs('model', exist_ok=True)
with open('model/class_indices.json', 'w') as f:
    json.dump(train_data.class_indices, f, indent=4)
print(f"\n✅ Loaded {train_data.num_classes} classes successfully!")

## 🏗️ Step 4: Build Model & Phase 1 Training (Frozen Backbone)
Train top classification head for 3 epochs.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.25),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='model/agrismart_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

print("🏋️ Phase 1: Training classification head (3 epochs)...")
history_p1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3,
    callbacks=[checkpoint]
)

## 🔬 Step 5: Phase 2 Fine-Tuning (Unfreeze Last 20 Layers)
Unfreeze the top 20 layers of MobileNetV2 and train with a gentle learning rate (`1e-5`).

In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("🚀 Phase 2: Fine-Tuning top 20 layers (8 epochs)...")
history_p2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=[checkpoint, early_stopping]
)

## 📊 Step 6: Compute Official Hackathon Metrics (Macro-F1 & Confusion Matrix)
Evaluates on the held-out test set and saves the confusion matrix and report.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Load best checkpoint
best_model = tf.keras.models.load_model('model/agrismart_model.keras', compile=False)

test_data.reset()
print("Running inference across held-out test set...")
probabilities = best_model.predict(test_data, verbose=1)
y_pred = probabilities.argmax(axis=1)
y_true = test_data.classes

accuracy = float((y_pred == y_true).mean())
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
matrix = confusion_matrix(y_true, y_pred)

print("\n" + "="*60)
print(f"🎯 FINAL ACCURACY : {accuracy:.2%}")
print(f"🏆 FINAL MACRO-F1 : {macro_f1:.2%}")
print("="*60)
print(report)

# Plot and save confusion matrix
os.makedirs('report/external_evaluation', exist_ok=True)
with open('report/external_evaluation/classification_report.txt', 'w') as f:
    f.write(f"Accuracy: {accuracy:.4f}\nMacro-F1: {macro_f1:.4f}\n\n{report}")

plt.figure(figsize=(16, 14))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title(f'AgriSmart AI Confusion Matrix (Macro-F1: {macro_f1:.2%})')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('report/external_evaluation/confusion_matrix.png', dpi=150)
plt.show()
print("✅ Saved report and confusion_matrix.png")

## 💾 Step 7: Download Model Weights for Local Repo
Run this cell to download the finished `agrismart_model.keras` directly to your local computer.

In [ ]:
if colab_env:
    from google.colab import files
    print("Downloading fine-tuned model...")
    files.download('model/agrismart_model.keras')
    files.download('report/external_evaluation/confusion_matrix.png')
    files.download('report/external_evaluation/classification_report.txt')
    print("✅ Downloads initiated!")
else:
    print("Running locally — model already saved at model/agrismart_model.keras")